# Compare CAA Vectors

This notebook loads two vectors (`CAA.pt` and a specific layer vector) and computes their cosine similarity and L2 distance.

In [ ]:
import torch
import torch.nn.functional as F
import os

# Paths
caa_path = "Vector/CAA/extracted/ai_risk_coordinate.pt"
caa_path = "Vector/CAST/extracted/llama_coordinate_no_pca.pt"
vector_path = "Vector/CAA/vectors/coordinate/vec_layer_13_Llama-2-7b-chat-hf.pt"

print(f"Checking files:")
print(f"CAA.pt exists: {os.path.exists(caa_path)}")
print(f"Vector file exists: {os.path.exists(vector_path)}")

Checking files:
CAA.pt exists: True
Vector file exists: True


In [18]:
def load_and_extract(path):
    print(f"Loading {path}...")
    try:
        data = torch.load(path, map_location='cpu')
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None

    if isinstance(data, torch.Tensor):
        print(f"  Loaded directly as Tensor. Shape: {data.shape}")
        return data

    if isinstance(data, dict):
        print(f"  Loaded as dict. Keys: {data.keys()}")
        # Try to find a likely vector key
        candidates = ['vector', 'direction', 'steering_vector', 'vec']
        for key in candidates:
            if key in data:
                print(f"  Found key '{key}'.")
                return data[key]
        
        # If the keys look like layer numbers, maybe we need to pick one? 
        # But for now let's check if there is only one value that is a tensor
        tensor_vals = [v for k, v in data.items() if isinstance(v, torch.Tensor)]
        if len(tensor_vals) == 1:
            print("  Found exactly one tensor value in dict, using that.")
            return tensor_vals[0]
        
        print("  Could not automatically identify vector tensor in dict.")
        return data
    
    print(f"  Unknown data type: {type(data)}")
    return data

# Load vectors
vec1_raw = load_and_extract(caa_path)[13]
vec2_raw = load_and_extract(vector_path)

Loading Vector/CAST/extracted/llama_coordinate_no_pca.pt...
  Loaded as dict. Keys: dict_keys(['steering_vector', 'metadata'])
  Found key 'steering_vector'.
Loading Vector/CAA/vectors/coordinate/vec_layer_13_Llama-2-7b-chat-hf.pt...
  Loaded directly as Tensor. Shape: torch.Size([4096])


/tmp/ipykernel_558389/567618888.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path, map_location='cpu')


In [20]:
# Preprocess and Compare
if isinstance(vec1_raw, torch.Tensor) and isinstance(vec2_raw, torch.Tensor):
    # Ensure float32
    v1 = vec1_raw.float()
    v2 = vec2_raw.float()

    # Flatten
    v1_flat = v1.view(-1)
    v2_flat = v2.view(-1)

    print(f"\nComparing flattened vectors:")
    print(f"Vector 1 size: {v1_flat.shape[0]}")
    print(f"Vector 2 size: {v2_flat.shape[0]}")

    if v1_flat.shape[0] != v2_flat.shape[0]:
        print("WARNING: Vectors have different sizes! Metrics may be invalid or fail.")
        # Crop to min length just to show something? Or error out.
        min_len = min(v1_flat.shape[0], v2_flat.shape[0])
        print(f"Truncating to {min_len} for comparison...")
        v1_flat = v1_flat[:min_len]
        v2_flat = v2_flat[:min_len]

    # Cosine Similarity
    # dim=0 because they are 1D vectors now
    cosine_sim = F.cosine_similarity(v1_flat.unsqueeze(0), v2_flat.unsqueeze(0)).item()
    print(f"Cosine Similarity: {cosine_sim:.6f}")

    # L2 Distance
    l2_dist = torch.norm(v1_flat - v2_flat, p=2).item()
    print(f"L2 Distance: {l2_dist:.6f}")
    
    #Norm
    norm1 = torch.norm(v1_flat, p=2).item()
    norm2 = torch.norm(v2_flat, p=2).item()
    print(f"Norm 1: {norm1:.6f}")
    print(f"Norm 2: {norm2:.6f}")
    
    # Dot product
    dot_prod = torch.dot(v1_flat, v2_flat).item()
    print(f"Dot Product: {dot_prod:.6f}")

    
else:
    print("Could not run comparison because one or both files did not yield a Tensor.")
    print(f"Vec1 type: {type(vec1_raw)}")
    print(f"Vec2 type: {type(vec2_raw)}")


Comparing flattened vectors:
Vector 1 size: 4096
Vector 2 size: 4096
Cosine Similarity: 0.831705
L2 Distance: 7.503860
Norm 1: 1.840800
Norm 2: 8.964933
Dot Product: 13.725336
